# 01 — Bronze: SCB

Downloads two tables from SCB's PxWebApi v2 and lands them unchanged in the
bronze volume, then builds bronze Delta tables from the landed files.

| Table | Content | Grain |
|---|---|---|
| TAB3277 | Newly registered passenger cars | region × fuel type × month |
| TAB3276 | Passenger cars in traffic | region × ownership category × year |

Bronze never calls the API directly into a table. Files land first, tables are
built from files. A re-run reparses local files instead of re-fetching.

In [0]:
CATALOG = "axenil_assignment1"
BRONZE_SCHEMA = "bronze"
VOLUME_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/raw"

TABLE_NEW_REGISTRATIONS = "TAB3277"
TABLE_CARS_IN_TRAFFIC = "TAB3276"

FIRST_YEAR = 2016
LAST_YEAR = 2025

API_BASE = "https://statistikdatabasen.scb.se/api/v2/tables"
REQUEST_HEADERS = {"User-Agent": "nackademin-laddstolpar-axel"}

In [0]:
import os
import time
import json
import itertools


def fetch_metadata(table_id):
    response = requests.get(
        f"{API_BASE}/{table_id}/metadata?lang=sv",
        headers=REQUEST_HEADERS,
        timeout=60,
    )
    response.raise_for_status()
    metadata = response.json()
    return {
        dimension_id: category_codes(metadata, dimension_id)
        for dimension_id in metadata["id"]
    }


def category_codes(payload, dimension_id):
    index_map = payload["dimension"][dimension_id]["category"]["index"]
    if isinstance(index_map, dict):
        return [code for code, _ in sorted(index_map.items(), key=lambda item: item[1])]
    return list(index_map)


def fetch_slice(table_id, parameters, file_path, maximum_attempts=5):
    if os.path.exists(file_path):
        return False

    for attempt in range(maximum_attempts):
        response = requests.get(
            f"{API_BASE}/{table_id}/data",
            params=parameters,
            headers=REQUEST_HEADERS,
            timeout=300,
        )
        if response.status_code in (403, 429):
            wait_seconds = 15 * (attempt + 1)
            print(f"  throttled ({response.status_code}), waiting {wait_seconds}s")
            time.sleep(wait_seconds)
            continue
        response.raise_for_status()

        temporary_path = file_path + ".tmp"
        with open(temporary_path, "wb") as output_file:
            output_file.write(response.content)
        os.replace(temporary_path, file_path)
        return True

    raise RuntimeError(f"gave up after {maximum_attempts} attempts: {file_path}")


def flatten_json_stat(payload):
    """Returns (column_names, rows) from a json-stat2 dataset."""
    dimension_ids = payload["id"]
    category_lists = [category_codes(payload, dimension_id) for dimension_id in dimension_ids]
    values = payload["value"]

    def value_at(position):
        if isinstance(values, dict):
            return values.get(str(position))
        return values[position]

    rows = [
        combination + (value_at(position),)
        for position, combination in enumerate(itertools.product(*category_lists))
    ]
    return dimension_ids + ["value"], rows

## Download

**json-stat2, not CSV.** The CSV output pivots time into columns, so each
year's file would have different column names. json-stat2 returns a flat value
array with a fixed dimension order, which flattens to one uniform long table.

**One call per year.** 315 regions × 8 fuel types × 12 months = 30,240 cells,
well inside the API limit of 150,000 cells per call. Per-year files also make
the download resumable.

**Files are skipped if they already exist.** This is the first of three
idempotency mechanisms in the pipeline: bronze skips, silver merges, gold
rebuilds. Re-running this notebook costs one metadata call and nothing else.

**Writes are atomic.** Each file is written to `.tmp` and renamed on success,
so an interrupted run can't leave a truncated file that the skip check would
later treat as complete.

In [0]:
import os
import time


def fetch_metadata(table_id):
    response = requests.get(
        f"{API_BASE}/{table_id}/metadata?lang=sv",
        headers=REQUEST_HEADERS,
        timeout=60,
    )
    response.raise_for_status()
    metadata = response.json()
    return {
        dimension_id: list(metadata["dimension"][dimension_id]["category"]["index"].keys())
        for dimension_id in metadata["id"]
    }


def fetch_slice(table_id, parameters, file_path, maximum_attempts=5):
    if os.path.exists(file_path):
        return False

    for attempt in range(maximum_attempts):
        response = requests.get(
            f"{API_BASE}/{table_id}/data",
            params=parameters,
            headers=REQUEST_HEADERS,
            timeout=300,
        )
        if response.status_code in (403, 429):
            wait_seconds = 15 * (attempt + 1)
            print(f"  throttled ({response.status_code}), waiting {wait_seconds}s")
            time.sleep(wait_seconds)
            continue
        response.raise_for_status()

        temporary_path = file_path + ".tmp"
        with open(temporary_path, "wb") as output_file:
            output_file.write(response.content)
        os.replace(temporary_path, file_path)
        return True

    raise RuntimeError(f"gave up after {maximum_attempts} attempts: {file_path}")

In [0]:
new_registration_values = fetch_metadata(TABLE_NEW_REGISTRATIONS)
new_registration_directory = f"{VOLUME_PATH}/{TABLE_NEW_REGISTRATIONS}"
os.makedirs(new_registration_directory, exist_ok=True)

for year in range(FIRST_YEAR, LAST_YEAR + 1):
    month_codes = [code for code in new_registration_values["Tid"] if code.startswith(str(year))]
    if not month_codes:
        continue

    parameters = {
        "lang": "sv",
        "outputFormat": "json-stat2",
        "valueCodes[Region]": "*",
        "valueCodes[Drivmedel]": "*",
        "valueCodes[ContentsCode]": "TK1001AA",
        "valueCodes[Tid]": ",".join(month_codes),
    }
    was_fetched = fetch_slice(
        TABLE_NEW_REGISTRATIONS,
        parameters,
        f"{new_registration_directory}/new_registrations_{year}.json",
    )
    print(year, "fetched" if was_fetched else "already present")
    if was_fetched:
        time.sleep(0.5)

## Bronze tables

Every column is stored as a string. Bronze should not fail because a source
value changes shape, and casting is silver's job. It also protects the leading
zeros in region codes — `0114` must not become `114`, since the first two
characters are the county code.

Three metadata columns are added for lineage: which SCB table the row came
from, which file it was parsed out of, and when it was ingested.

**Row count is verified against the source, not hardcoded.** Each json-stat
payload declares its own dimension sizes, and their product is how many cells
that file contains. The loader asserts that the flattened row count matches.
This stays correct when SCB publishes a new year, when the configured period
changes, and when a year is only partially published.

It does not detect a *missing* file — that is a coverage check, and belongs in
`05_kvalitetskontroller`.

In [0]:
from pyspark.sql import functions


def load_into_bronze(source_directory, table_name, source_table_id):
    all_rows = []
    column_names = None

    for file_name in sorted(os.listdir(source_directory)):
        if not file_name.endswith(".json"):
            continue
        with open(f"{source_directory}/{file_name}", "r", encoding="utf-8") as input_file:
            payload = json.load(input_file)
        column_names, rows = flatten_json_stat(payload)
        all_rows.extend(row + (source_table_id, file_name) for row in rows)

    dataframe = (
        spark.createDataFrame(all_rows, column_names + ["_source_table", "_source_file"])
             .withColumn("value", functions.col("value").cast("string"))
             .withColumn("_ingested_at", functions.current_timestamp())
    )
    (dataframe.write
              .mode("overwrite")
              .option("overwriteSchema", "true")
              .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"))
    return dataframe


load_into_bronze(new_registration_directory, "scb_new_registrations", TABLE_NEW_REGISTRATIONS)
load_into_bronze(cars_in_traffic_directory, "scb_cars_in_traffic", TABLE_CARS_IN_TRAFFIC)

In [0]:
display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.scb_new_registrations").limit(20))